# Module 04 — Thời gian thực thi lệnh CPU & dung lượng bộ nhớ / CPU Timing & Memory Capacity
**AE2.021 · USTH**

🇻🇳 Notebook tính thời gian thực thi lệnh theo T-state/tần số xung nhịp, tính dung lượng bộ nhớ cần thiết
từ số lượng IC DRAM, và mô phỏng đơn giản chu trình fetch-decode-execute.

🇬🇧 This notebook computes instruction execution time from T-states/clock frequency, computes required
memory capacity from the number of DRAM chips, and simulates a simple fetch-decode-execute cycle.

## 1. Thời gian thực thi lệnh: t = n_T × (1/f_clk)
🎯 **Phương pháp này trả lời câu hỏi gì?** Biết tần số xung nhịp và số T-state một lệnh cần, làm sao suy
ra chính xác thời gian thực thi thực tế tính bằng giây — và kiểm chứng lại câu hỏi gốc (50MHz, 11 T-state).

In [1]:
def exec_time(n_tstates: int, f_clk_hz: float) -> float:
    T = 1.0 / f_clk_hz
    return n_tstates * T

# Cau hoi goc (Tooley Ch.7 Q15): 50MHz, 11 T-state -> ky vong 220ns
t = exec_time(11, 50e6)
print(f"50MHz, 11 T-state -> t = {t*1e9:.1f} ns  (ky vong 220 ns)")

# Cau tu sinh: 20MHz, 4 T-state -> ky vong 200ns
t2 = exec_time(4, 20e6)
print(f"20MHz, 4 T-state  -> t = {t2*1e9:.1f} ns  (ky vong 200 ns)")

print()
for f_mhz in (10, 20, 50, 100):
    print(f"f_clk={f_mhz:3d} MHz -> 1 T-state = {1/(f_mhz*1e6)*1e9:.1f} ns")

50MHz, 11 T-state -> t = 220.0 ns  (ky vong 220 ns)
20MHz, 4 T-state  -> t = 200.0 ns  (ky vong 200 ns)

f_clk= 10 MHz -> 1 T-state = 100.0 ns
f_clk= 20 MHz -> 1 T-state = 50.0 ns
f_clk= 50 MHz -> 1 T-state = 20.0 ns
f_clk=100 MHz -> 1 T-state = 10.0 ns


#### 📤 Đầu ra thật
Hai kết quả tính được (220ns và 200ns) khớp chính xác với đáp án đã công bố trên trang web cho câu hỏi
gốc và câu hỏi tự sinh tương ứng — xác nhận công thức t = n_T × (1/f_clk) và cách áp dụng là đúng.

## 2. Dung lượng bộ nhớ từ số lượng IC DRAM
🎯 **Phương pháp này trả lời câu hỏi gì?** Có N byte cần lưu và mỗi IC DRAM chỉ chứa được (rows×cols) bit,
cần tối thiểu bao nhiêu IC — và địa chỉ hoá được bao nhiêu ô nhớ với một bus địa chỉ n-bit cho trước.

In [2]:
def chips_needed(total_bytes: int, chip_words: int, chip_bits_per_word: int) -> int:
    chip_bytes = chip_words * chip_bits_per_word // 8
    return -(-total_bytes // chip_bytes)  # lam tron len

# Cau hoi goc (Tooley Ch.6 Q13): can bao nhieu IC 16K x 4-bit de co 32K byte?
n_chips = chips_needed(32*1024, 16*1024, 4)
print(f"32KB tu IC 16Kx4bit -> can {n_chips} IC (ky vong 4)")

def addressable_locations(address_bus_bits: int) -> int:
    return 2 ** address_bus_bits

for bits in (8, 16, 24, 32):
    n = addressable_locations(bits)
    print(f"Bus dia chi {bits:2d}-bit -> dia chi hoa duoc {n:>12,} o nho "
          f"(hex lon nhat = {n-1:X})")

32KB tu IC 16Kx4bit -> can 4 IC (ky vong 4)
Bus dia chi  8-bit -> dia chi hoa duoc          256 o nho (hex lon nhat = FF)
Bus dia chi 16-bit -> dia chi hoa duoc       65,536 o nho (hex lon nhat = FFFF)
Bus dia chi 24-bit -> dia chi hoa duoc   16,777,216 o nho (hex lon nhat = FFFFFF)
Bus dia chi 32-bit -> dia chi hoa duoc 4,294,967,296 o nho (hex lon nhat = FFFFFFFF)


#### 📤 Đầu ra thật
`chips_needed` cho ra đúng 4 — khớp với câu hỏi gốc Tooley Ch.6 Q13. Với bus 24-bit, địa chỉ hex lớn nhất
là FFFFFF — khớp với câu hỏi gốc Ch.6 Q5. Với bus 32-bit, số ô nhớ địa chỉ hoá được là 4.294.967.296 = 4GB
— khớp với câu hỏi tự sinh thêm.

## 3. Mô phỏng đơn giản chu trình fetch–decode–execute
🎯 **Phương pháp này trả lời câu hỏi gì?** Minh hoạ cụ thể từng bước PC trỏ tới lệnh, lệnh được nạp vào IR,
giải mã, rồi thực thi — thay vì chỉ mô tả bằng lời.

In [3]:
program = {
    0: ("LOAD", "A", 5),   # A = 5
    1: ("LOAD", "B", 3),   # B = 3
    2: ("ADD",  "A", "B"), # A = A + B
    3: ("HALT", None, None),
}

registers = {"A": 0, "B": 0}
PC = 0

def fetch(pc):
    return program[pc]

def execute(instr):
    op, x, y = instr
    if op == "LOAD":
        registers[x] = y
    elif op == "ADD":
        registers[x] = registers[x] + registers[y]
    elif op == "HALT":
        return False
    return True

running = True
while running:
    instr = fetch(PC)               # FETCH
    print(f"PC={PC}  IR={instr}   (fetch)")
    op = instr[0]                   # DECODE (o day don gian: doc truc tiep opcode)
    running = execute(instr)        # EXECUTE
    print(f"        -> sau khi thuc thi: registers={registers}")
    PC += 1

print("\nKet qua cuoi:", registers)

PC=0  IR=('LOAD', 'A', 5)   (fetch)
        -> sau khi thuc thi: registers={'A': 5, 'B': 0}
PC=1  IR=('LOAD', 'B', 3)   (fetch)
        -> sau khi thuc thi: registers={'A': 5, 'B': 3}
PC=2  IR=('ADD', 'A', 'B')   (fetch)
        -> sau khi thuc thi: registers={'A': 8, 'B': 3}
PC=3  IR=('HALT', None, None)   (fetch)
        -> sau khi thuc thi: registers={'A': 8, 'B': 3}

Ket qua cuoi: {'A': 8, 'B': 3}


#### 📤 Đầu ra thật
Sau 4 chu kỳ fetch-decode-execute, thanh ghi A = 8 (5+3), đúng với chương trình đã định nghĩa — minh hoạ
trực quan PC tăng dần từng bước, mỗi lệnh được fetch trước khi được thực thi, đúng thứ tự chu trình lệnh
đã học ở phần lý thuyết.

In [4]:
print("\n" + "="*70, "\nFloyd Ch.11 - Section 11-1 Semiconductor Memory Basics (Problem 4)\n" + "="*70)
print("4) Dia chi bo nho (0-256) tuong ung moi so hex:")
for h in ["0C","5E","DF"]:
    print(f"   {h}16 = {int(h,16)} (thap phan)")

print("\n" + "="*70, "\nFloyd Ch.11 - Problem 2 (khai niem, khong tinh so): quan he bit/byte/nibble/word\n" + "="*70)
print("   1 nibble = 4 bit. 1 byte = 8 bit = 2 nibble. 1 word = boi so cua byte (thuong 16/32/64-bit = 2/4/8 byte),")
print("   tuy kien truc CPU cu the quy dinh do rong 1 word.")

print("\n" + "="*70, "\nFloyd Ch.11 - Section 11-1 Problem 5 (RAM tinh 4 hang, dieu kien cho truoc)\n" + "="*70)
rows = [0,0,0,0]
conditions = [(0,1,1),(1,0,1),(2,1,0),(3,0,1)]  # (row_index, row_select, data_in_bit)
print("   Ban dau ca 4 hang deu = 0. Ap dung dieu kien (Row select=1 moi ghi):")
for row_idx, sel, data in conditions:
    if sel == 1:
        rows[row_idx] = data
        print(f"   Row {row_idx} duoc chon (select=1), ghi Data in={data} -> Row {row_idx} = {data}")
    else:
        print(f"   Row {row_idx} KHONG duoc chon (select=0) -> giu nguyen = {rows[row_idx]}")
print("   Noi dung cuoi cung 4 hang:", rows)



Floyd Ch.11 - Section 11-1 Semiconductor Memory Basics (Problem 4)
4) Dia chi bo nho (0-256) tuong ung moi so hex:
   0C16 = 12 (thap phan)
   5E16 = 94 (thap phan)
   DF16 = 223 (thap phan)

Floyd Ch.11 - Problem 2 (khai niem, khong tinh so): quan he bit/byte/nibble/word
   1 nibble = 4 bit. 1 byte = 8 bit = 2 nibble. 1 word = boi so cua byte (thuong 16/32/64-bit = 2/4/8 byte),
   tuy kien truc CPU cu the quy dinh do rong 1 word.

Floyd Ch.11 - Section 11-1 Problem 5 (RAM tinh 4 hang, dieu kien cho truoc)
   Ban dau ca 4 hang deu = 0. Ap dung dieu kien (Row select=1 moi ghi):
   Row 0 duoc chon (select=1), ghi Data in=1 -> Row 0 = 1
   Row 1 KHONG duoc chon (select=0) -> giu nguyen = 0
   Row 2 duoc chon (select=1), ghi Data in=0 -> Row 2 = 0
   Row 3 KHONG duoc chon (select=0) -> giu nguyen = 0
   Noi dung cuoi cung 4 hang: [1, 0, 0, 0]
